In [17]:
import geopandas as gpd
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import duckdb
import matplotlib.pyplot as plt
plt.style.use("ggplot")

DATA = Path("data")
SHORE_PATH = DATA / "parcels_shore.parquet"

# Schema — no data loaded
schema = pq.read_schema(SHORE_PATH)
print(f"Rows (all years): {pq.read_metadata(SHORE_PATH).num_rows:,}")
print(f"Columns: {len(schema)}\n")
for field in schema:
    print(f"  {field.name:<45} {field.type}")

FileNotFoundError: [Errno 2] Failed to open local file 'data/parcels_shore.parquet'. Detail: [errno 2] No such file or directory

In [ ]:
df = duckdb.sql("""
    SELECT mod_iv_recordid,
                mod_iv_county_name,
                record_id,
                property_class,
                building_description, 
                sale_price,
                year_constructed,
                land_value,
                improvement_value,
                net_taxable_value,
               # census_tract,
               # census_block,
                sale_year,
                sale_year_1,
                sale_price_2020,
                bd_stories,
                bd_material,
                bd_style,
                bd_garage_stalls,
                bd_garage_type,
                bd_is_condo,
                bd_is_common_element,
                bd_unit_floor,
                bd_is_model_plan,
                FLD_ZONE,
                ZONE_SUBTY,
                flood_risk,
                dist_to_ocean_mi,
                census_tract_geoid,
                pct_seasonal,
                geometry
    
    FROM read_parquet('data/parcels_shore.parquet')
""").df()



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
print(df.shape)
print(df.info())

(6779800, 31)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6779800 entries, 0 to 6779799
Data columns (total 31 columns):
 #   Column                Dtype  
---  ------                -----  
 0   mod_iv_recordid       object 
 1   mod_iv_county_name    object 
 2   record_id             object 
 3   property_class        object 
 4   building_description  object 
 5   sale_price            object 
 6   year_constructed      object 
 7   land_value            object 
 8   improvement_value     object 
 9   net_taxable_value     object 
 10  census_tract          object 
 11  census_block          object 
 12  sale_year             Int64  
 13  sale_year_1           Int64  
 14  sale_price_2020       float64
 15  bd_stories            float64
 16  bd_material           object 
 17  bd_style              object 
 18  bd_garage_stalls      float64
 19  bd_garage_type        object 
 20  bd_is_condo           bool   
 21  bd_is_common_element  bool   
 22  bd_unit_floor         object

In [ ]:
df.head()

,mod_iv_recordid,mod_iv_county_name,record_id,property_class,building_description,sale_price,year_constructed,land_value,improvement_value,net_taxable_value,...,bd_is_common_element,bd_unit_floor,bd_is_model_plan,FLD_ZONE,ZONE_SUBTY,flood_risk,dist_to_ocean_mi,census_tract_geoid,pct_seasonal,geometry
0,010304700 00141 20,ATLANTIC,20,2,3S-F-G,379900,1987,363700,36300,400000,...,False,None,False,AE,None,High,0.731198,34001010105,0.474446,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
1,010304700 00142 20,ATLANTIC,20,2,3S-F-G,271000,1987,328400,46600,375000,...,False,None,False,AE,None,High,0.725009,34001010105,0.474446,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
2,010304700 00143 20,ATLANTIC,20,2,3S-F-G,90000,1987,328400,128700,457100,...,False,None,False,AE,None,High,0.721095,34001010105,0.474446,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
3,010304700 00144 20,ATLANTIC,20,2,3S-F-G,320000,1987,246300,73700,320000,...,False,None,False,AE,None,High,0.717173,34001010105,0.474446,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."
4,010304700 00145 20,ATLANTIC,20,2,3S-F-G,160000,1987,328400,134800,463200,...,False,None,False,AE,None,High,0.713264,34001010105,0.474446,"[1, 6, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 0, 1, ..."


In [ ]:
# check for nulls
nulls = df.isnull().sum()
print(nulls[nulls > 0])

building_description     305024
year_constructed         876695
census_tract            6735391
census_block            6735290
sale_year                549616
sale_year_1              549616
sale_price_2020          564301
bd_stories               879935
bd_material             1464326
bd_style                5630733
bd_garage_stalls        4240686
bd_garage_type          4240686
bd_unit_floor           6680741
FLD_ZONE                  79875
ZONE_SUBTY              2761288
flood_risk                73369
dist_to_ocean_mi          73369
census_tract_geoid       114169
pct_seasonal             301285
dtype: int64


In [ ]:
df = duckdb.sql(f"""
    SELECT 
        mod_iv_recordid,
        mod_iv_year,
        mod_iv_county_name,
        record_id,
        property_class,
        building_description, 
        CAST(sale_price AS DOUBLE) AS sale_price,
        CAST(year_constructed AS INTEGER) AS year_constructed,
        CAST(land_value AS DOUBLE) AS land_value,
        CAST(improvement_value AS DOUBLE) AS improvement_value,
        CAST(net_taxable_value AS DOUBLE) AS net_taxable_value,
        sale_year,
        sale_year_1,
        sale_price_2020,
        bd_stories,
        bd_material,
        bd_style,
        bd_garage_stalls,
        bd_garage_type,
        bd_is_condo,
        bd_is_common_element,
        bd_unit_floor,
        bd_is_model_plan,
        FLD_ZONE,
        ZONE_SUBTY,
        flood_risk,
        dist_to_ocean_mi,
        census_tract_geoid,
        pct_seasonal,
        geometry
    FROM '{SHORE_PATH}'
    WHERE 
        CAST(sale_price AS DOUBLE) > 25000
        AND CAST(year_constructed AS INTEGER) > 1800 AND CAST(year_constructed AS INTEGER) <= 2025
        AND flood_risk IS NOT NULL
        AND dist_to_ocean_mi IS NOT NULL
        AND pct_seasonal IS NOT NULL
        AND census_tract_geoid IS NOT NULL
        AND geometry IS NOT NULL
        AND mod_iv_year >= 2015
""").df()



In [ ]:
df.shape

(3223292, 30)

In [ ]:
# export cleaned data for modeling
df.to_parquet(DATA / "parcels_shore_clean.parquet", index=False)